# Práctica 4. Seguimiento, conteo y CSV de entrega

Un único recorrido del vídeo que produce a la vez:

1. vídeo anotado con cajas, IDs de seguimiento y matrículas,
2. `detecciones.csv` con una fila por objeto y fotograma,
3. conteo de objetos **únicos** por clase (IDs distintos, no detecciones),
4. recortes de la mejor matrícula de cada vehículo, listos para el OCR de la práctica 4b.

Flujo: pasada 1 (`track()` de personas y vehículos con ByteTrack + detector de matrículas) → **unión de pistas** para corregir los cambios de ID → pasada 2 (vídeo, CSV y recortes con los IDs ya unificados) → conteo.

In [8]:
import csv
import os
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

## Configuración

In [9]:
VIDEO = Path("../datos/videos/C0142.MP4")          # vídeo a procesar (cambiar por el de test)
PESOS_MATRICULAS = Path("runs/matriculas_yolo11n_lejanas/weights/best.pt")
PESOS_OBJETOS = "yolo11n.pt"

SALIDA = Path("resultado_seguimiento")
SALIDA.mkdir(exist_ok=True)
VIDEO_OUT = SALIDA / "seguimiento.mp4"
CSV_OUT = SALIDA / "detecciones.csv"
CONTEO_OUT = SALIDA / "conteo.csv"
RECORTES = SALIDA / "matriculas"
RECORTES.mkdir(exist_ok=True)

# Clases COCO de interés
CLASES = {0: "person", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
VEHICULOS = {"car", "motorcycle", "bus", "truck"}
# Clases que el detector puede confundir entre sí; sirven para unir pistas de un mismo objeto
GRUPO = {"person": "persona", "motorcycle": "moto", "car": "vehiculo", "truck": "vehiculo", "bus": "vehiculo"}

IMGSZ_OBJETOS = 1280    # resolución de inferencia para personas y vehículos
CONF_OBJETOS = 0.10     # BAJO a propósito: ByteTrack necesita las detecciones débiles para recuperar pistas ocluidas
IMGSZ_MATRICULAS = 1920  # resolución completa, para no encoger las matrículas lejanas
CONF_MATRICULAS = 0.25

# --- Unión de pistas (re-identificación a posteriori) ---
MAX_HUECO = 300     # fotogramas máximos que un objeto puede estar perdido (oclusión) y seguir siendo el mismo
IOU_MIN = 0.30      # solape mínimo entre la última caja de la pista perdida y la primera de la nueva
MIN_FRAMES = 10     # un objeto cuenta solo si aparece al menos en tantos fotogramas

MARGEN_RECORTE = 0.10  # margen relativo alrededor de la matrícula al guardar el recorte

SHOW = False                                            # True para ver el vídeo mientras se procesa
MAX_FRAMES = int(os.environ.get("MAX_FRAMES", 0)) or None  # None = vídeo completo

assert VIDEO.exists(), f"No existe el vídeo: {VIDEO.resolve()}"
assert PESOS_MATRICULAS.exists(), f"No existen los pesos: {PESOS_MATRICULAS.resolve()}"

# Configuración propia de ByteTrack: memoria larga y umbrales más permisivos que los de por defecto
TRACKER = SALIDA / "bytetrack_custom.yaml"
TRACKER.write_text(
    "tracker_type: bytetrack\n"
    "track_high_thresh: 0.30\n"   # detecciones fuertes: primera asociación
    "track_low_thresh: 0.10\n"    # detecciones débiles: segunda asociación (recuperan oclusiones)
    "new_track_thresh: 0.40\n"    # confianza mínima para abrir una pista nueva (menos pistas espurias)
    "track_buffer: 150\n"         # fotogramas que una pista perdida se conserva antes de borrarse
    "match_thresh: 0.85\n"
    "fuse_score: True\n"
)

148

In [10]:
modelo_objetos = YOLO(PESOS_OBJETOS)
modelo_matriculas = YOLO(str(PESOS_MATRICULAS))
print("Matrículas:", modelo_matriculas.names)

Matrículas: {0: 'License Plate'}


## Asociación matrícula-vehículo

Una matrícula detectada se asigna al vehículo que la contiene. Se exigen las mismas restricciones geométricas que en
`probar_pesos_video.ipynb` (relación de aspecto de matrícula, zona baja del vehículo, tamaño razonable). Si varias
matrículas caen en el mismo vehículo, se conserva la de mayor confianza. Las matrículas que no encajan en ningún
vehículo se descartan como falsos positivos.

In [11]:
def cabe_en_vehiculo(caja, veh):
    """Comprueba que la caja tiene forma de matrícula y está en la parte baja del vehículo `veh`."""
    x1, y1, x2, y2 = caja
    ancho, alto = x2 - x1, y2 - y1
    if alto <= 0 or not 1.2 <= ancho / alto <= 6.5:
        return False
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    vx1, vy1, vx2, vy2 = veh
    v_ancho, v_alto = vx2 - vx1, vy2 - vy1
    return (vx1 <= cx <= vx2
            and vy1 + 0.35 * v_alto <= cy <= vy2 + 0.05 * v_alto
            and ancho <= 0.65 * v_ancho)


def asociar_matriculas(placas, vehiculos):
    """
    placas:    lista de (caja, conf)
    vehiculos: lista de (track_id, caja)
    Devuelve {track_id: (caja_matricula, conf)} con la mejor matrícula de cada vehículo.
    Si una matrícula cabe en varios vehículos (solapados), se asigna al de menor área.
    """
    asignadas = {}
    for caja, conf in placas:
        candidatos = [(tid, v) for tid, v in vehiculos if cabe_en_vehiculo(caja, v)]
        if not candidatos:
            continue
        tid, _ = min(candidatos, key=lambda tv: (tv[1][2] - tv[1][0]) * (tv[1][3] - tv[1][1]))
        if tid not in asignadas or conf > asignadas[tid][1]:
            asignadas[tid] = (caja, conf)
    return asignadas

## Pasada 1: detección y seguimiento

Se recorre el vídeo y se guardan en memoria las detecciones con el ID bruto del tracker, sin dibujar nada todavía.
`persist=True` mantiene los IDs entre llamadas a `track`.

**Por qué dos pasadas.** Cuando un coche cruza por delante de otros aparcados, el tracker pierde a los aparcados y,
al reaparecer, les da un ID nuevo (*ID switch*). Para corregirlo hace falta ver el vídeo entero antes de decidir
qué IDs son el mismo objeto, y solo entonces dibujar y escribir el CSV con los IDs ya unificados.

In [12]:
objs_por_frame = {}      # fotograma -> [(id_bruto, clase, conf, caja)]
placas_por_frame = {}    # fotograma -> [(caja, conf)]

vid = cv2.VideoCapture(str(VIDEO))
total = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))
n = 0
while True:
    ret, img = vid.read()
    if not ret or (MAX_FRAMES and n >= MAX_FRAMES):
        break
    n += 1

    r = modelo_objetos.track(img, persist=True, imgsz=IMGSZ_OBJETOS, conf=CONF_OBJETOS,
                             classes=list(CLASES), tracker=str(TRACKER), verbose=False)[0]
    objs = []
    if r.boxes.id is not None:   # es None mientras el tracker no ha confirmado ninguna pista
        for tid, cls, conf, caja in zip(r.boxes.id.int().tolist(), r.boxes.cls.int().tolist(),
                                        r.boxes.conf.tolist(), r.boxes.xyxy.tolist()):
            objs.append((tid, CLASES[cls], conf, caja))
    objs_por_frame[n] = objs

    p = modelo_matriculas(img, imgsz=IMGSZ_MATRICULAS, conf=CONF_MATRICULAS, verbose=False)[0].boxes
    placas_por_frame[n] = list(zip(p.xyxy.tolist(), p.conf.tolist()))

    if n % 100 == 0:
        print(f"{n}/{total} fotogramas")

vid.release()
N_FRAMES = n
ids_brutos = {t for objs in objs_por_frame.values() for t, *_ in objs}
print(f"Pasada 1 terminada: {N_FRAMES} fotogramas, {len(ids_brutos)} IDs brutos del tracker")

100/2832 fotogramas
200/2832 fotogramas
300/2832 fotogramas
400/2832 fotogramas
500/2832 fotogramas
600/2832 fotogramas
700/2832 fotogramas
800/2832 fotogramas
900/2832 fotogramas
1000/2832 fotogramas
1100/2832 fotogramas
1200/2832 fotogramas
1300/2832 fotogramas
1400/2832 fotogramas
1500/2832 fotogramas
1600/2832 fotogramas
1700/2832 fotogramas
1800/2832 fotogramas
1900/2832 fotogramas
2000/2832 fotogramas
2100/2832 fotogramas
2200/2832 fotogramas
2300/2832 fotogramas
2400/2832 fotogramas
2500/2832 fotogramas
2600/2832 fotogramas
2700/2832 fotogramas
2800/2832 fotogramas
Pasada 1 terminada: 2832 fotogramas, 247 IDs brutos del tracker


## Unión de pistas (re-identificación a posteriori)

Cada ID bruto es una *pista* (*tracklet*) con un fotograma inicial, uno final, y una primera y última caja. Una
pista **B** se une a una pista anterior **A** cuando:

- son del mismo grupo de clase (persona, moto o vehículo de cuatro ruedas, para tolerar confusiones coche/camión),
- **A** termina antes de que empiece **B**, con un hueco de como mucho `MAX_HUECO` fotogramas,
- la última caja de **A** y la primera de **B** se solapan (IoU ≥ `IOU_MIN`),
- **A** no tiene ya otra sucesora.

Es justo el caso de un coche aparcado que queda tapado por otro que pasa: reaparece en el mismo sitio.
Las cadenas de uniones se resuelven con una estructura de conjuntos disjuntos y cada objeto conserva el ID de su
primera pista.

In [13]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def unir_pistas(objs_por_frame, max_hueco, iou_min):
    """Devuelve {id_bruto: id_unificado}."""
    pistas = {}
    for fr in sorted(objs_por_frame):
        for tid, clase, _, caja in objs_por_frame[fr]:
            p = pistas.setdefault(tid, {"clases": Counter(), "ini": fr, "caja_ini": caja})
            p["clases"][clase] += 1
            p["fin"], p["caja_fin"] = fr, caja
    for p in pistas.values():
        p["grupo"] = GRUPO[p["clases"].most_common(1)[0][0]]

    padre = {t: t for t in pistas}
    con_sucesora = set()
    for b in sorted(pistas, key=lambda t: pistas[t]["ini"]):
        pb = pistas[b]
        mejor = None
        for a, pa in pistas.items():
            if (a == b or a in con_sucesora or pa["grupo"] != pb["grupo"]
                    or not 0 < pb["ini"] - pa["fin"] <= max_hueco):
                continue
            s = iou(pa["caja_fin"], pb["caja_ini"])
            if s >= iou_min and (mejor is None or s > mejor[0]):
                mejor = (s, a)
        if mejor:
            con_sucesora.add(mejor[1])
            padre[b] = mejor[1]

    def raiz(t):
        while padre[t] != t:
            t = padre[t]
        return t
    return {t: raiz(t) for t in pistas}


mapa_id = unir_pistas(objs_por_frame, MAX_HUECO, IOU_MIN)
print(f"IDs brutos: {len(mapa_id)}  ->  objetos tras unir pistas: {len(set(mapa_id.values()))}")

IDs brutos: 247  ->  objetos tras unir pistas: 166


## Pasada 2: vídeo, CSV y recortes con los IDs unificados

Se vuelve a leer el vídeo y se dibuja, se escribe el CSV y se guardan los recortes de matrícula usando el ID
unificado. Antes se calcula, para cada objeto, su clase (voto mayoritario), el número de fotogramas en que aparece
y el primer fotograma, para decidir si cuenta (`MIN_FRAMES`) y para el contador en pantalla.

In [14]:
CAMPOS = ["fotograma", "tipo_objeto", "confianza", "id_tracking", "x1", "y1", "x2", "y2",
          "matricula_detectada", "confianza_matricula", "mx1", "my1", "mx2", "my2", "texto_matricula"]

COLORES = {"person": (0, 200, 0), "car": (255, 128, 0), "motorcycle": (0, 200, 255),
           "bus": (200, 0, 200), "truck": (0, 0, 255)}

# Estadísticas por objeto unificado
frames_por_id = Counter()
clases_por_id = defaultdict(Counter)
primer_frame = {}
trayectoria = defaultdict(list)
for fr in sorted(objs_por_frame):
    for tid, clase, _, (x1, y1, x2, y2) in objs_por_frame[fr]:
        u = mapa_id[tid]
        frames_por_id[u] += 1
        clases_por_id[u][clase] += 1
        primer_frame.setdefault(u, fr)
        trayectoria[u].append(((x1 + x2) / 2, (y1 + y2) / 2))

validos = {u for u, c in frames_por_id.items() if c >= MIN_FRAMES}
clase_de = {u: clases_por_id[u].most_common(1)[0][0] for u in frames_por_id}
mejor_matricula = {}   # id -> (conf, recorte)

vid = cv2.VideoCapture(str(VIDEO))
fps = vid.get(cv2.CAP_PROP_FPS) or 25
ancho = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(str(VIDEO_OUT), cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))

with open(CSV_OUT, "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=CAMPOS)
    escritor.writeheader()
    for n in range(1, N_FRAMES + 1):
        ret, img = vid.read()
        if not ret:
            break

        # solo se muestran/guardan los objetos que cuentan (descarta pistas fugaces)
        objetos = [(mapa_id[t], clase, conf, caja) for t, clase, conf, caja in objs_por_frame[n]
                   if mapa_id[t] in validos]
        vehiculos = [(tid, caja) for tid, clase, _, caja in objetos if clase in VEHICULOS]
        matriculas = asociar_matriculas(placas_por_frame[n], vehiculos)

        for tid, clase, conf, (x1, y1, x2, y2) in objetos:
            clase = clase_de[tid]   # clase estable por objeto, para que no parpadee entre coche y camión
            fila = {"fotograma": n, "tipo_objeto": clase, "confianza": round(conf, 3), "id_tracking": tid,
                    "x1": round(x1), "y1": round(y1), "x2": round(x2), "y2": round(y2),
                    "matricula_detectada": 0, "confianza_matricula": "", "mx1": "", "my1": "",
                    "mx2": "", "my2": "", "texto_matricula": ""}   # el texto lo rellena el OCR (4b)

            color = COLORES[clase]
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            cv2.putText(img, f"{clase} #{tid} {conf:.2f}", (int(x1), max(int(y1) - 6, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            if tid in matriculas:
                (mx1, my1, mx2, my2), mconf = matriculas[tid]
                fila.update({"matricula_detectada": 1, "confianza_matricula": round(mconf, 3),
                             "mx1": round(mx1), "my1": round(my1), "mx2": round(mx2), "my2": round(my2)})
                cv2.rectangle(img, (int(mx1), int(my1)), (int(mx2), int(my2)), (255, 0, 255), 2)
                cv2.putText(img, f"{mconf:.2f}", (int(mx1), max(int(my1) - 6, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 2)

                # guarda el recorte de mayor confianza de cada vehículo (entrada del OCR)
                if tid not in mejor_matricula or mconf > mejor_matricula[tid][0]:
                    mw, mh = mx2 - mx1, my2 - my1
                    cx1 = max(int(mx1 - MARGEN_RECORTE * mw), 0)
                    cy1 = max(int(my1 - MARGEN_RECORTE * mh), 0)
                    cx2 = min(int(mx2 + MARGEN_RECORTE * mw), ancho)
                    cy2 = min(int(my2 + MARGEN_RECORTE * mh), alto)
                    mejor_matricula[tid] = (mconf, img[cy1:cy2, cx1:cx2].copy())

            escritor.writerow(fila)

        # contador en pantalla: objetos válidos que han aparecido hasta este fotograma
        hasta_ahora = Counter(clase_de[u] for u in validos if primer_frame[u] <= n)
        texto = "  ".join(f"{c}: {hasta_ahora.get(c, 0)}" for c in CLASES.values())
        cv2.rectangle(img, (0, 0), (ancho, 34), (0, 0, 0), -1)
        cv2.putText(img, texto, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        out.write(img)
        if SHOW:
            cv2.imshow("Seguimiento", img)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

vid.release()
out.release()
cv2.destroyAllWindows()
print("Vídeo:", VIDEO_OUT.resolve())
print("CSV:  ", CSV_OUT.resolve())

Vídeo: D:\CarpetaDescargasC\Documentos\AaCurso27\VC\Practica4\VC_P4\P4\resultado_seguimiento\seguimiento.mp4
CSV:   D:\CarpetaDescargasC\Documentos\AaCurso27\VC\Practica4\VC_P4\P4\resultado_seguimiento\detecciones.csv


## Conteo por clase

Se cuentan **objetos únicos tras la unión de pistas**, no detecciones ni IDs brutos. Cada objeto se asigna a la clase
que más veces se le ha detectado y solo se cuenta si aparece en al menos `MIN_FRAMES` fotogramas. Aun así el
conteo es una estimación: dos objetos distintos que se cruzan pueden fusionarse, y una oclusión muy larga puede
dejar un objeto partido en dos. Conviene comprobar el vídeo a mano y comentar estos errores en el README.

In [15]:
conteo = Counter(clase_de[u] for u in validos)

print(f"IDs brutos del tracker: {len(ids_brutos)}")
print(f"Objetos tras unir pistas: {len(frames_por_id)}  |  con >= {MIN_FRAMES} fotogramas: {len(validos)}")
for clase in CLASES.values():
    print(f"{clase:>12}: {conteo.get(clase, 0)}")

with open(CONTEO_OUT, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["clase", "total_objetos_unicos"])
    for clase in CLASES.values():
        w.writerow([clase, conteo.get(clase, 0)])

for viejo in RECORTES.glob("*.jpg"):   # evita mezclar recortes de ejecuciones anteriores
    viejo.unlink()
guardados = 0
for tid, (mconf, recorte) in mejor_matricula.items():
    if recorte.size:
        cv2.imwrite(str(RECORTES / f"id{tid:04d}_{mconf:.2f}.jpg"), recorte)
        guardados += 1
print(f"Recortes de matrícula guardados: {guardados} en {RECORTES.resolve()}")

IDs brutos del tracker: 247
Objetos tras unir pistas: 166  |  con >= 10 fotogramas: 149
      person: 51
         car: 93
  motorcycle: 2
         bus: 2
       truck: 1
Recortes de matrícula guardados: 41 en D:\CarpetaDescargasC\Documentos\AaCurso27\VC\Practica4\VC_P4\P4\resultado_seguimiento\matriculas


## Extra: flujo direccional

Se compara el centro de cada objeto al principio y al final de su recorrido. Solo se consideran objetos válidos
con un desplazamiento mínimo (`UMBRAL`, en píxeles) para ignorar los que están prácticamente quietos.

In [16]:
UMBRAL = 80

def direccion(puntos):
    (x0, y0), (x1, y1) = puntos[0], puntos[-1]
    dx, dy = x1 - x0, y1 - y0
    if max(abs(dx), abs(dy)) < UMBRAL:
        return "estático"
    if abs(dx) >= abs(dy):
        return "derecha" if dx > 0 else "izquierda"
    return "abajo (se acerca)" if dy > 0 else "arriba (se aleja)"

flujo = defaultdict(Counter)
for u in validos:
    flujo[clase_de[u]][direccion(trayectoria[u])] += 1

for clase, c in flujo.items():
    print(clase, dict(c))

bus {'izquierda': 1, 'estático': 1}
car {'izquierda': 37, 'estático': 47, 'derecha': 9}
person {'derecha': 19, 'estático': 25, 'izquierda': 7}
motorcycle {'izquierda': 2}
truck {'estático': 1}
